# Use Case 6: ACID Transactions & Concurrency Control

**The Concept:** 
When multiple agents or services write to the same database simultaneously, data corruption can occur without proper isolation. **ACID transactions** guarantee that operations are Atomic, Consistent, Isolated, and Durable.

**The Architecture:** 
SochDB implements **Serializable Snapshot Isolation (SSI)** — the gold standard for concurrent transaction safety. It uses an append-only MVCC engine with conflict detection. Transactions return **Hybrid Logical Clock (HLC)** timestamps for causal ordering.

---

### Step 0: Install Packages

In [1]:
!pip install sochdb

import json
import time

You should consider upgrading via the '/Users/sushanth/sochdb_python/venv/bin/python3 -m pip install --upgrade pip' command.


### Step 1: Initialize Database
Open an embedded database for our transaction demos.

In [2]:
from sochdb import Database

db = Database.open("./transactions_demo_db")
print("Database opened for transaction demos.")

Database opened for transaction demos.


### Step 2: Basic Key-Value Operations
SochDB is fundamentally an embedded KV store. Let's do basic put/get/delete operations.

In [3]:
# Simple KV operations
db.put(b"user:alice:balance", b"1000")
db.put(b"user:bob:balance", b"500")

alice_balance = db.get(b"user:alice:balance")
bob_balance = db.get(b"user:bob:balance")

print(f"Alice's balance: {alice_balance.decode()}")
print(f"Bob's balance: {bob_balance.decode()}")
print(f"Key exists? {db.exists(b'user:alice:balance')}")

Alice's balance: 1000
Bob's balance: 500
Key exists? True


### Step 3: Atomic Transactions (Context Manager)
Transfer 200 from Alice to Bob **atomically**. Either both the debit and credit happen, or neither does.

In [4]:
# Atomic fund transfer using context manager
with db.transaction() as txn:
    # Read current balances
    alice_bal = int(txn.get(b"user:alice:balance").decode())
    bob_bal = int(txn.get(b"user:bob:balance").decode())
    
    transfer_amount = 200
    
    # Debit Alice, Credit Bob
    txn.put(b"user:alice:balance", str(alice_bal - transfer_amount).encode())
    txn.put(b"user:bob:balance", str(bob_bal + transfer_amount).encode())
    
    # Transaction commits automatically when exiting the `with` block

# Verify
print(f"Alice's balance after transfer: {db.get(b'user:alice:balance').decode()}")
print(f"Bob's balance after transfer: {db.get(b'user:bob:balance').decode()}")

Alice's balance after transfer: 800
Bob's balance after transfer: 700


### Step 4: Transaction Rollback on Error
If an error occurs mid-transaction, all changes are automatically rolled back.

In [5]:
# Simulate a failed transaction
try:
    with db.transaction() as txn:
        txn.put(b"user:alice:balance", b"0")  # Would set Alice to 0
        
        # Simulate an error before commit
        raise ValueError("Insufficient funds! Aborting transaction.")
        
except ValueError as e:
    print(f"Transaction aborted: {e}")

# Alice's balance should be unchanged (800 from previous transfer)
print(f"Alice's balance (unchanged): {db.get(b'user:alice:balance').decode()}")

Transaction aborted: Insufficient funds! Aborting transaction.
Alice's balance (unchanged): 800


### Step 5: Manual Transaction Control
For more control, you can manually `begin`, `commit`, or `abort` transactions.

In [6]:
txn = db.begin_transaction()

# Perform some writes
txn.put(b"user:carol:balance", b"750")
txn.put(b"user:dave:balance", b"250")

# Commit returns an HLC (Hybrid Logical Clock) timestamp for causal ordering
hlc_timestamp = txn.commit()
print(f"Transaction committed at HLC timestamp: {hlc_timestamp}")

# Verify
print(f"Carol's balance: {db.get(b'user:carol:balance').decode()}")
print(f"Dave's balance: {db.get(b'user:dave:balance').decode()}")

Transaction committed at HLC timestamp: 11
Carol's balance: 750
Dave's balance: 250


### Step 6: Batch Operations
Efficiently insert/read/delete multiple keys at once.

In [7]:
# Batch put
items = [
    (b"config:max_retries", b"3"),
    (b"config:timeout_ms", b"5000"),
    (b"config:debug_mode", b"false"),
    (b"config:version", b"2.1.0"),
]
count = db.put_batch(items)
print(f"Batch inserted {count} keys.")

# Batch get
keys = [b"config:max_retries", b"config:timeout_ms", b"config:debug_mode", b"config:version"]
values = db.get_batch(keys)
print("\nBatch read results:")
for k, v in zip(keys, values):
    print(f"  {k.decode()} = {v.decode() if v else 'None'}")

Batch inserted 4 keys.

Batch read results:
  config:max_retries = 3
  config:timeout_ms = 5000
  config:debug_mode = false
  config:version = 2.1.0


### Step 7: Prefix Scanning
Scan all keys matching a prefix — useful for retrieving all config values, all user records, etc.

In [8]:
# Scan all keys starting with "config:"
config_entries = db.scan_prefix(b"config:")
print("All configuration entries:")
for entry in config_entries:
    print(f"  {entry}")

print()

# Scan all user balances
user_entries = db.scan_prefix(b"user:")
print("All user balance entries:")
for entry in user_entries:
    print(f"  {entry}")

All configuration entries:
  (b'config:debug_mode', b'false')
  (b'config:max_retries', b'3')
  (b'config:timeout_ms', b'5000')
  (b'config:version', b'2.1.0')

All user balance entries:
  (b'user:alice:balance', b'800')
  (b'user:bob:balance', b'700')
  (b'user:carol:balance', b'750')
  (b'user:dave:balance', b'250')


### Step 8: Path-Based Hierarchical Storage
SochDB supports storing data in a file-system-like hierarchy using path keys.

In [10]:
# Store data using hierarchical paths
db.put_path("/apps/chatbot/config", json.dumps({"model": "gemini-3-flash", "temperature": 0.7}).encode())
db.put_path("/apps/chatbot/prompts/system", b"You are a helpful assistant.")
db.put_path("/apps/chatbot/prompts/greeting", b"Hello! How can I help you today?")
db.put_path("/apps/search/config", json.dumps({"index": "hybrid", "top_k": 10}).encode())

# Retrieve a specific path
config = json.loads(db.get_path("/apps/chatbot/config").decode())
print(f"Chatbot config: {config}")

# Retrieve all known paths under /apps/chatbot/
print("\nAll paths under /apps/chatbot/:")
chatbot_paths = [
    "/apps/chatbot/config",
    "/apps/chatbot/prompts/system",
    "/apps/chatbot/prompts/greeting",
]
for path in chatbot_paths:
    val = db.get_path(path)
    if val:
        print(f"  {path} → {val.decode()}")

Chatbot config: {'model': 'gemini-3-flash', 'temperature': 0.7}

All paths under /apps/chatbot/:
  /apps/chatbot/config → {"model": "gemini-3-flash", "temperature": 0.7}
  /apps/chatbot/prompts/system → You are a helpful assistant.
  /apps/chatbot/prompts/greeting → Hello! How can I help you today?


### Step 9: SQL Within a Transaction
Execute SQL queries inside a transaction for atomic multi-statement workflows.

In [12]:
with db.transaction() as txn:
    txn.execute("CREATE TABLE IF NOT EXISTS audit_log (id INT, action TEXT, timestamp INT)")
    txn.execute("INSERT INTO audit_log (id, action, timestamp) VALUES (1, 'transfer_200_alice_to_bob', 1709600000)")
    txn.execute("INSERT INTO audit_log (id, action, timestamp) VALUES (2, 'create_user_carol', 1709600100)")
    txn.execute("INSERT INTO audit_log (id, action, timestamp) VALUES (3, 'create_user_dave', 1709600200)")

# Query outside the transaction
result = db.execute("SELECT * FROM audit_log ORDER BY timestamp")
print("Audit Log:")
print(f"Columns: {result.columns}")
for row in result.rows:
    print(f"  {row}")

Audit Log:
Columns: ['id', 'action', 'timestamp']
  {'id': 1, 'action': 'transfer_200_alice_to_bob', 'timestamp': 1709600000}
  {'id': 2, 'action': 'create_user_carol', 'timestamp': 1709600100}
  {'id': 3, 'action': 'create_user_dave', 'timestamp': 1709600200}


### Cleanup

In [ ]:
db.close()
print("Database closed.")